# SECTION 1: IMPORTS AND SETUP

In [1]:
pip install numpy pandas matplotlib nltk scikit-learn scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\quratulain.adnan\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [3]:
import numpy as np
import pandas as pd
import re
import string
import nltk

import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

# Evaluation
from sklearn.metrics import (
    f1_score,
    confusion_matrix,
    adjusted_rand_score,
    normalized_mutual_info_score
)

# Download NLTK resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Setup complete.")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\quratulain.adnan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\quratulain.adnan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\quratulain.adnan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\quratulain.adnan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\quratulain.adnan\AppData\Roaming\nltk_data...


Setup complete.


[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\quratulain.adnan\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\quratulain.adnan\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


# Section 2: Dataset Loading

In [8]:
from sklearn.datasets import fetch_20newsgroups

# using 6 categories ==> WSD is computationally expensive.
#   - enough diversity for interesting clustering
#   - Small enough for WSD/lexical chains to run in reasonable time
CATEGORIES = [
    'rec.sport.hockey',
    'sci.space',
    'talk.politics.guns',
    'comp.graphics',
    'rec.autos',
    'sci.med',
]

# Number of clusters = number of categories we chose
# k-Means will find exactly this many clusters
NUM_CLUSTERS = len(CATEGORIES)   # = 6

# subset='all'  → use both train and test splits combined
# remove=...    → strip headers, footers, and quoted replies
print("Loading 20 Newsgroups dataset...")
newsgroups = fetch_20newsgroups(
    subset='all',
    categories=CATEGORIES,
    remove=('headers', 'footers', 'quotes'),
    random_state=42          
)

# newsgroups.data  → list of raw document strings
# newsgroups.target → array of integer class labels (0, 1, 2, ...)
# newsgroups.target_names → the human-readable category names

# WSD is slow. limit each category to MAX_PER_CLASS documents.
# keeps the dataset balanced AND fast.
import numpy as np

MAX_PER_CLASS = 100

np.random.seed(42)
selected_indices = []
for class_id in range(NUM_CLUSTERS):
    # find all indices belonging to this class
    class_indices = np.where(newsgroups.target == class_id)[0]
    # Randomly pick at most MAX_PER_CLASS of them
    
    chosen = np.random.choice(
        class_indices,
        size=min(MAX_PER_CLASS, len(class_indices)),
        replace=False   # no duplicates
    )
    selected_indices.extend(chosen.tolist())   # will contain around 600 docs

# shuffle the selected indices so classes aren't in blocks
np.random.shuffle(selected_indices)

# final document list and label list 
documents   = [newsgroups.data[i]   for i in selected_indices]
true_labels = [newsgroups.target[i] for i in selected_indices]

print(f"\nCategories selected : {CATEGORIES}")
print(f"Total documents     : {len(documents)}")
print(f"Total labels        : {len(true_labels)}")
print(f"Documents per class : {MAX_PER_CLASS} (max)")
print(f"\nSample document (first 300 chars):\n")
print(documents[0][:300])
print("\nCorresponding label:", newsgroups.target_names[true_labels[0]])

Loading 20 Newsgroups dataset...

Categories selected : ['rec.sport.hockey', 'sci.space', 'talk.politics.guns', 'comp.graphics', 'rec.autos', 'sci.med']
Total documents     : 600
Total labels        : 600
Documents per class : 100 (max)

Sample document (first 300 chars):


Here goes:

More than a few years back (if you were born that year, you can legally drink),
we tried it out.  We found an 8 ft. deep cistern that we lined with some 10 ft.
2X6s.  We put a large can (one of those industrial sized pork'n beans cans)
stuffed with oily rags and scraps of wood in the bo

Corresponding label: talk.politics.guns
